In [2]:
!git clone https://github.com/speechbrain/speechbrain/

Cloning into 'speechbrain'...
Updating files:  73% (1056/1442)
Updating files:  74% (1068/1442)
Updating files:  75% (1082/1442)
Updating files:  76% (1096/1442)
Updating files:  77% (1111/1442)
Updating files:  78% (1125/1442)
Updating files:  79% (1140/1442)
Updating files:  80% (1154/1442)
Updating files:  81% (1169/1442)
Updating files:  82% (1183/1442)
Updating files:  83% (1197/1442)
Updating files:  84% (1212/1442)
Updating files:  85% (1226/1442)
Updating files:  86% (1241/1442)
Updating files:  87% (1255/1442)
Updating files:  88% (1269/1442)
Updating files:  89% (1284/1442)
Updating files:  90% (1298/1442)
Updating files:  91% (1313/1442)
Updating files:  92% (1327/1442)
Updating files:  93% (1342/1442)
Updating files:  94% (1356/1442)
Updating files:  95% (1370/1442)
Updating files:  96% (1385/1442)
Updating files:  97% (1399/1442)
Updating files:  98% (1414/1442)
Updating files:  99% (1428/1442)
Updating files: 100% (1442/1442)
Updating files: 100% (1442/1442), done.


In [3]:
%cd speechbrain
!pip install -r requirements.txt
!pip install -e .

c:\Users\Ons Hadrich\Desktop\PFE\Tacotron2\tts-arabic-pytorch\speakerRecognizer\speechbrain


c:\Users\Ons Hadrich\Desktop\PFE\Tacotron2\tts-arabic-pytorch\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


  Using cached isort-5.13.2-py3-none-any.whl.metadata (12 kB)
  Using cached HyperPyYAML-1.2.2-py3-none-any.whl.metadata (7.6 kB)
  Using cached sentencepiece-0.2.0-cp310-cp310-win_amd64.whl.metadata (8.3 kB)
  Using cached torchaudio-2.6.0-cp310-cp310-win_amd64.whl.metadata (6.7 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached mypy_extensions-1.0.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached pathspec-0.12.1-py3-none-any.whl.metadata (21 kB)
  Using cached tomli-2.2.1-py3-none-any.whl.metadata (10 kB)
  Using cached pluggy-1.5.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached ruamel.yaml-0.18.10-py3-none-any.whl.metadata (23 kB)
  Using cached regex-2024.11.6-cp310-cp310-win_amd64.whl.metadata (41 kB)
  Using cached ruamel.yaml.clib-0.2.12-cp310-cp310-win_amd64.whl.metadata (2.8 kB)
  Using cached distlib-0.3.9-py2.py3-none-any.whl.metadata (5.2 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------

In [12]:
import os
import pickle
import torch
import torchaudio
import pandas as pd
from speechbrain.inference.speaker import EncoderClassifier
from sklearn.metrics.pairwise import cosine_similarity
import shutil

# Charger le modèle pré-entraîné de reconnaissance de locuteur
classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")

# Initialiser les dictionnaires pour stocker les données des locuteurs
speaker_database = {}  # Clé: fichier audio, Valeur: ID du locuteur
speaker_ids = {}       # Clé: ID du locuteur, Valeur: embedding

# Définir le dossier pour stocker les fichiers des locuteurs
speakers_dir = "data\custom_data\speaker"
os.makedirs(speakers_dir, exist_ok=True)

# Fonction pour extraire l'embedding d'un fichier audio
def extract_speaker_embedding(audio_path):
    signal, fs = torchaudio.load(audio_path)  # Charger le fichier audio
    embedding = classifier.encode_batch(signal)  # Extraire l'embedding
    return embedding[0].detach().cpu().numpy().flatten()  # Convertir en tableau 1D

# Fonction pour attribuer ou créer un ID de locuteur basé sur l'embedding
def get_or_create_speaker_id(embedding, threshold=0.5):
    for speaker_id, stored_embedding in speaker_ids.items():
        similarity = cosine_similarity([embedding], [stored_embedding])[0][0]
        if similarity > threshold:  # Si suffisamment similaire
            return speaker_id
    # Créer un nouvel ID, en commençant par 0
    new_speaker_id = len(speaker_ids)  # Commence à 0 pour le premier locuteur
    speaker_ids[new_speaker_id] = embedding
    return new_speaker_id

# Fonction pour traiter le dataset à partir d'un fichier CSV
def process_dataset_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    data = []

    for index, row in df.iterrows():
        audio_file = row['Nom du fichier']
        diacritized_text = row['Texte corrigé']  # Récupérer le texte diacritisé

        if audio_file.endswith('.wav'):
            print(f"Traitement du fichier : {audio_file}")

            # Extraire l'embedding et obtenir l'ID du locuteur
            embedding = extract_speaker_embedding(audio_file)
            speaker_id = get_or_create_speaker_id(embedding)

            # Ajouter à la base de données
            speaker_database[audio_file] = speaker_id
            # Ajouter les trois colonnes à la liste de données
            data.append([audio_file, diacritized_text, speaker_id])

            # Copier le fichier dans le dossier des locuteurs
            speaker_audio_file = os.path.join(speakers_dir, f"speaker{speaker_id}.wav")
            if not os.path.exists(speaker_audio_file):  # Éviter les doublons
                shutil.copy(audio_file, speaker_audio_file)
                print(f"Fichier copié : {audio_file} -> {speaker_audio_file}")
            else:
                print(f"Le locuteur {speaker_id} existe déjà, fichier ignoré.")

    # Sauvegarder les données
    with open('data\checkpoints\locuteurs_data.pkl', 'wb') as f:
        pickle.dump(speaker_database, f)
    with open('data\checkpoints\locuteurs_ID.pkl', 'wb') as f:
        pickle.dump(speaker_ids, f)

    # Créer un nouveau DataFrame avec les trois colonnes
    df_new = pd.DataFrame(data, columns=["Nom du fichier", "Texte diacrité", "Speaker_ID"])
    output_csv_path = 'data\custom_data\MSATTS.csv'
    df_new.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Données sauvegardées dans : {output_csv_path}")

# Lancer le traitement

csv_path = 'data\custom_data\dataset_v1.csv'
process_dataset_from_csv(csv_path)

# Charger et afficher les résultats
with open('data\checkpoints\locuteurs_data.pkl', 'rb') as f:
    loaded_speaker_database = pickle.load(f)
with open('data\checkpoints\locuteurs_ID.pkl', 'rb') as f:
    loaded_speaker_ids = pickle.load(f)

print("Base de données des locuteurs :", loaded_speaker_database)
print("IDs des locuteurs :", loaded_speaker_ids)

FileNotFoundError: [Errno 2] No such file or directory: 'data\\custom_data\\dataset_v1.csv'